# 01 · Baseline And Protocol

Notebook central para fijar una sola vez el protocolo de evaluación, el esquema de exportación de resultados y los baselines contra los que se comparan LGWR/GWR, SAR, RF + Kriging y GNN.

## Objetivos

- Definir un split reproducible y justo.
- Elegir métricas comunes.
- Crear una función estándar para exportar artefactos por modelo.
- Dejar uno o dos baselines simples para calibrar expectativas.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output" / "01_baseline_and_protocol"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

## Dataset de referencia

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "rental_listings_model_input.parquet"
# df = pd.read_parquet(DATA_PATH)

target_col = "price_m2"
coord_cols = ["lon", "lat"]
feature_cols = []  # completar con las features comunes a todos los modelos
group_col = None    # por ejemplo barrio/comuna si queres analizar sesgos territoriales

## Protocolo de split

Elegí una estrategia y dejala fija. Si más adelante cambiás el split, hacelo acá y no por notebook.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
VALID_SIZE = 0.2

# Ejemplo simple. Reemplazar si usas split espacial o temporal.
# train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE)
# train_df, valid_df = train_test_split(train_df, test_size=VALID_SIZE, random_state=RANDOM_STATE)
# split_summary = {"train": len(train_df), "valid": len(valid_df), "test": len(test_df)}
# split_summary

## Métricas estándar

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    residuals = y_true - y_pred
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "bias": float(residuals.mean()),
        "median_abs_error": float(np.median(np.abs(residuals))),
    }

## Formato común de artefactos

Cada notebook de modelo debería exportar al menos:

- `metrics.json`
- `test_predictions.parquet`
- `residuals.parquet`
- `interpretability.parquet` o un artefacto equivalente
- `run_config.json`

In [ ]:
def export_model_artifacts(model_name, metrics, test_frame, interpretability_frame=None, run_config=None):
    model_dir = PROJECT_ROOT / "notebooks" / "output" / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    (model_dir / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
    test_frame.to_parquet(model_dir / "test_predictions.parquet", index=False)

    residual_cols = [col for col in ["y_true", "y_pred", "residual", "split"] if col in test_frame.columns]
    if residual_cols:
        test_frame[residual_cols + [c for c in [group_col] if c and c in test_frame.columns]].to_parquet(
            model_dir / "residuals.parquet", index=False
        )

    if interpretability_frame is not None:
        interpretability_frame.to_parquet(model_dir / "interpretability.parquet", index=False)

    if run_config is not None:
        (model_dir / "run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False))

## Baselines tentativos

In [ ]:
# X_train = train_df[feature_cols]
# y_train = train_df[target_col]
# X_test = test_df[feature_cols]
# y_test = test_df[target_col]

# dummy = DummyRegressor(strategy="median")
# dummy.fit(X_train, y_train)
# dummy_pred = dummy.predict(X_test)
# regression_metrics(y_test, dummy_pred)

## Checklist para todos los modelos

- Misma transformación del target.
- Mismas observaciones evaluadas.
- Mismo set de features base cuando aplique.
- Misma definición de train/valid/test.
- Exportación de artefactos bajo el formato estándar.